In [ ]:
import sqlite3
import sys
from pathlib import Path
import pandas as pd

# 1. Connect to SQLite
db_path = Path.cwd().parent.parent / "data" / "processed" / "reddit-data-300926.db"
conn = sqlite3.connect(db_path)

sample_posts = pd.read_sql("SELECT id, title, description FROM posts ORDER BY RANDOM() LIMIT 60", conn)
sample_posts.to_csv("evaluation_sentiment_testset.csv", index=False)

In [ ]:
import sys
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd
from IPython.display import display


# Find the project root from the notebook's current directory.
PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "src" / "verifier" / "decomposer.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate src/verifier/decomposer.py. "
        "Open this notebook inside your project repository."
    )

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from verifier.decomposer import decompose_post


DB_PATH = (
    PROJECT_ROOT / "data" / "processed" / "reddit-data-300926.db"
)

OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "Prototype"
OUTPUT_PATH = OUTPUT_DIR / "evaluation_claims_testset.csv"

if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Output:", OUTPUT_PATH)

In [ ]:
query = """
SELECT
    ID AS post_id,
    COALESCE(TITLE, '') AS title,
    COALESCE(DESCRIPTION, '') AS description
FROM posts
WHERE FLAIR IS NULL OR FLAIR NOT IN ('Comedy', 'Meme')
ORDER BY ID;
"""

with closing(sqlite3.connect(DB_PATH)) as conn:
    posts_df = pd.read_sql_query(query, conn)

if posts_df.empty:
    raise ValueError("No eligible posts found in the database.")

print(f"Loaded {len(posts_df):,} eligible posts.")
display(posts_df.head())

In [ ]:
claim_rows = []
post_report = []

for post in posts_df.itertuples(index=False):
    extracted_claims = list(
        decompose_post(
            post.post_id,
            post.title,
            post.description,
        )
    )

    seen_texts = set()
    retained_count = 0

    for claim in extracted_claims:
        claim_text = claim.get("claim_text")

        if not isinstance(claim_text, str) or not claim_text.strip():
            raise ValueError(
                f"Invalid claim_text returned for post {post.post_id}: {claim!r}"
            )

        claim_text = claim_text.strip()
        duplicate_key = " ".join(claim_text.split())

        if duplicate_key in seen_texts:
            continue

        seen_texts.add(duplicate_key)
        retained_count += 1

        claim_rows.append({
            "claim_id": f"{post.post_id}_claim_{retained_count:03d}",
            "post_id": post.post_id,
            "title": post.title,
            "description": post.description,
            "claim_text": claim_text,
            "true_veracity": "",
            "evidence_source": "",
            "annotation_notes": "",
        })

    post_report.append({
        "post_id": post.post_id,
        "title": post.title,
        "extracted_claim_count": len(extracted_claims),
        "retained_claim_count": retained_count,
    })


columns = [
    "claim_id",
    "post_id",
    "title",
    "description",
    "claim_text",
    "true_veracity",
    "evidence_source",
    "annotation_notes",
]

claims_df = pd.DataFrame(claim_rows, columns=columns)
post_report_df = pd.DataFrame(post_report)

if claims_df.empty:
    raise ValueError("The decomposer returned no usable claims.")

assert claims_df["claim_id"].is_unique

print(f"Posts processed: {len(posts_df):,}")
print(f"Claims retained: {len(claims_df):,}")
print(
    "Posts with no retained claims:",
    int((post_report_df["retained_claim_count"] == 0).sum()),
)

display(claims_df[["claim_id", "claim_text", "true_veracity"]].head(20))

# Randomly select up to 50 claims, reproducibly.
claims_df = (
    claims_df.sample(
        n=min(50, len(claims_df)),
        random_state=42,
    )
    .reset_index(drop=True)
)

print(f"Selected {len(claims_df)} claims for evaluation.")
display(claims_df[["claim_id", "claim_text", "true_veracity"]])

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"{OUTPUT_PATH.name} already exists. "
        "Do not overwrite manual labels. "
        "Choose a new OUTPUT_PATH if you want another extraction."
    )

claims_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
    mode="x",
)

print(f"Saved {len(claims_df):,} claims to:")
print(OUTPUT_PATH)

display(
    post_report_df.loc[
        post_report_df["retained_claim_count"] == 0,
        ["post_id", "title"],
    ]
)